## LangChain Agents - A Comprehensive Guide

### What are Agents?

**Agents** are one of the most powerful concepts in LangChain. Unlike simple chains that follow a predetermined sequence, agents use an LLM as a **reasoning engine** to dynamically determine:
1. **Which actions** to perform
2. **In what order** to perform them
3. **When to stop** and return a final answer

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Agent** | The decision-maker that uses an LLM to reason about what to do next |
| **Tools** | Functions that agents can call to interact with the external world |
| **Compiled graph** | What `create_agent()` returns - a LangGraph state machine that runs the model/tool loop |
| **Message state** | The running list of messages (human, AI, tool) that carries context through a run |

### The Agent Loop

Every agent follows the same basic loop:
1. **Model call**: the LLM sees the conversation plus the tool schemas
2. **Tool call**: if the model requests a tool, the graph executes it
3. **Observation**: the tool's result is appended as a `ToolMessage`
4. **Repeat** until the model replies without requesting a tool

> ⚠️ **A note on ReAct**: LangChain 0.x drove this loop with *prompt engineering* - the model was asked to emit `Thought: / Action: / Action Input:` text that LangChain then regex-parsed. That is where `OutputParserException` came from. On LangChain 1.x, `create_agent()` uses the model's **native tool-calling API** instead, so there is no text to parse and nothing to fail parsing.

> 📘 This notebook uses `create_agent` (LangChain 1.x). The legacy `initialize_agent` / `AgentType` API still exists in the `langchain-classic` package but has been deprecated since 0.1 and receives no fixes.

---

## Section 1: Understanding Tools

### What are Tools?

**Tools** are functions that agents can use to interact with the external world. They extend the agent's capabilities beyond just generating text.

### Types of Tools

| Type | Examples | Use Case |
|------|----------|----------|
| **Built-in Tools** | `llm-math`, `wikipedia`, `serpapi` | Common utilities provided by LangChain |
| **Custom Tools** | Your own functions | Domain-specific operations |
| **Chain Tools** | Other LangChain chains | Complex multi-step operations |
| **Agent Tools** | Other agents | Hierarchical agent systems |

### Tool Components

Every tool has:
- **name**: A unique identifier for the tool
- **description**: Explains what the tool does (crucial for the agent to decide when to use it!)
- **func**: The actual function that gets executed

> 💡 **Pro Tip**: The `description` is extremely important! The agent uses it to decide which tool to use. Make it clear and specific.

In [2]:
# =============================================================================
# STEP 1: Environment Setup
# =============================================================================
# Load environment variables from .env file
# This typically includes API keys like OPENAI_API_KEY
# The find_dotenv() function searches for .env file in parent directories

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # Returns True if .env file was found and loaded

True

In [ ]:
# =============================================================================
# STEP 2: Loading Built-in Tools
# =============================================================================
# LangChain provides several pre-built tools that you can use out of the box
# Here we're loading the "llm-math" tool - a calculator that uses an LLM
#
# NOTE (LangChain 1.x): load_tools now lives in langchain_community.
# `from langchain.agents import load_tools` raises ImportError on 1.x.

from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_openai import ChatOpenAI

# Initialize the LLM that will power our agent
# gpt-4o-mini is a cost-effective model suitable for agent tasks
llm = ChatOpenAI(model="gpt-4o-mini")

# Load built-in tools by name
# "llm-math" - Uses an LLM to solve math problems by generating Python expressions
# Other available tools: "wikipedia", "serpapi", "requests", "terminal", etc.
tool_names = ["llm-math"]

# Some tools require an LLM to function (like llm-math which translates
# natural language math problems into executable expressions)
tools = load_tools(tool_names, llm=llm)

# Let's inspect what we got - a list of Tool objects
tools

[Tool(name='Calculator', description='Useful for when you need to answer questions about math.', func=<bound method Chain.run of LLMMathChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Translate a math problem into a expression that can be executed using Python\'s numexpr library. Use the output of running this code to answer the question.\n\nQuestion: ${{Question with math problem.}}\n```text\n${{single line mathematical expression that solves the problem}}\n```\n...numexpr.evaluate(text)...\n```output\n${{Output of running the code}}\n```\nAnswer: ${{Answer}}\n\nBegin.\n\nQuestion: What is 37593 * 67?\n```text\n37593 * 67\n```\n...numexpr.evaluate("37593 * 67")...\n```output\n2518731\n```\nAnswer: 2518731\n\nQuestion: 37593^(1/5)\n```text\n37593**(1/5)\n```\n...numexpr.evaluate("37593**(1/5)")...\n```output\n8.222831614237718\n```\nAnswer: 8.222831614237718\n\nQuestion: {question

In [ ]:
# =============================================================================
# STEP 3: Creating an Agent with create_agent
# =============================================================================
# create_agent() is the LangChain 1.x way to build a tool-calling agent.
# It returns a *compiled LangGraph graph*, not an AgentExecutor.

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

agent = create_agent(
    llm,                        # The LLM to use for reasoning (or a string like "openai:gpt-4o-mini")
    tools,                      # List of tools the agent can use
    system_prompt=(
        "You are a helpful assistant. "
        "Use the tools available to you whenever they help answer the question."
    ),
    # Middleware replaces the old max_iterations / handle_parsing_errors kwargs.
    # thread_limit caps model calls per conversation thread; exit_behavior="end"
    # returns whatever the agent has so far instead of raising.
    middleware=[ModelCallLimitMiddleware(thread_limit=3, exit_behavior="end")],
)


# =============================================================================
# Helper: the create_agent equivalent of verbose=True
# =============================================================================
# initialize_agent(verbose=True) printed the reasoning chain for you.
# With a graph you get something better: stream the run and print each step.

def run_agent(agent, question, config=None):
    """Stream an agent run, pretty-printing each new message as it arrives."""
    final_state = None
    for state in agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
        stream_mode="values",
    ):
        state["messages"][-1].pretty_print()
        final_state = state
    return final_state


# =============================================================================
# What changed from initialize_agent:
# =============================================================================
#   agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION  -> gone; native tool calling always
#   verbose=True                                 -> stream the graph (see run_agent)
#   max_iterations=3                             -> ModelCallLimitMiddleware
#   handle_parsing_errors=True                   -> unnecessary; no text parsing
#   memory=...                                   -> checkpointer= (see Section 3)
# =============================================================================

In [ ]:
# =============================================================================
# Inspecting the Agent's Components
# =============================================================================
# An AgentExecutor had a nested .agent.llm_chain.llm chain of attributes.
# A compiled graph exposes its structure instead - which is far more useful.

print("Returned type :", type(agent).__name__)
print("Graph nodes   :", list(agent.get_graph().nodes))
print("Model         :", llm.model_name)
print("Tools         :", [t.name for t in tools])

Returned type : CompiledStateGraph
Graph nodes   : ['__start__', 'model', 'tools', 'ModelCallLimitMiddleware.before_model', 'ModelCallLimitMiddleware.after_model', '__end__']
Model         : gpt-4o-mini
Tools         : ['Calculator']


In [9]:
# =============================================================================
# Visualizing the Agent Graph
# =============================================================================
# On 0.x you would print agent.agent.llm_chain.prompt.template to see the ReAct
# prompt. There is no such template now - the "prompt" is just your system_prompt
# plus the tool schemas, which the model receives through its tool-calling API.
#
# What's worth inspecting instead is the graph itself.

try:
    print(agent.get_graph().draw_ascii())   # requires the `grandalf` package
except Exception as exc:
    print(f"(ASCII render unavailable: {exc})\n")
    g = agent.get_graph()
    print("Nodes:", list(g.nodes))
    for edge in g.edges:
        print(f"  {edge.source} -> {edge.target}")

# =============================================================================
# READING THE GRAPH:
# =============================================================================
#   __start__ -> model    : the entry point
#   model -> tools        : taken when the model requests a tool call
#   tools -> model        : the tool result goes back for another model call
#   model -> __end__      : taken when the model answers without a tool call
# =============================================================================

                               +-----------+                                  
                               | __start__ |                                  
                               +-----------+                                  
                                      *                                       
                                      *                                       
                                      *                                       
                 +---------------------------------------+                    
                 | ModelCallLimitMiddleware.before_model |                    
                 +---------------------------------------+                    
                ......          ..        ..          .......                 
          ......              ..            ..               ......           
    ......                   .                ..                   ......     
....                   +-------+                .   

### Understanding the Agent Loop

On LangChain 0.x, the ReAct prompt template *was* the agent: a long instruction block asking the model to emit `Thought` / `Action` / `Action Input` / `Observation` text, which LangChain then parsed with regexes. Every parse failure was a runtime error.

On 1.x there is no template to print. The model receives:
1. Your `system_prompt` (plain text, exactly as you wrote it)
2. The conversation so far, as messages
3. The **tool schemas**, through the provider's native function-calling API

The model replies with a structured `tool_calls` field rather than text to be parsed. The graph reads that field, runs the tool, appends a `ToolMessage`, and loops.

Run the cell above to see the graph that implements this loop:

### Example 1: Agent Using Its Knowledge (No Tool Needed)

Let's ask the agent a question that doesn't require the calculator tool. Watch how the agent reasons and decides it already knows the answer:


In [10]:
# =============================================================================
# Running the Agent - General Knowledge Question
# =============================================================================
# Notice in the streamed output:
# - The agent recognizes this is a general knowledge question
# - It doesn't request the Calculator tool
# - It answers directly in a single model call

result = run_agent(agent, "How many members does the A Team have?")

# The result is the final graph state, a dict whose "messages" key holds the
# full conversation: HumanMessage -> (ToolMessage)* -> AIMessage
result["messages"][-1].content

================================ Human Message =================================

How many members does the A Team have?
================================== Ai Message ==================================

The A-Team, originally a fictional group from the television series "The A-Team" which aired from 1983 to 1987, typically consists of four main members:

1. John "Hannibal" Smith - the leader and strategist.
2. Templeton "Faceman" Peck - the con artist and supply specialist.
3. B.A. Baracus - the mechanic and strongman.
4. H.M. Murdock - the helicopter pilot and comic relief.

These four characters are the core members of The A-Team.
================================== Ai Message ==================================

The A-Team, originally a fictional group from the television series "The A-Team" which aired from 1983 to 1987, typically consists of four main members:

1. John "Hannibal" Smith - the leader and strategist.
2. Templeton "Faceman" Peck - the con artist and supply specialist.
3

'The A-Team, originally a fictional group from the television series "The A-Team" which aired from 1983 to 1987, typically consists of four main members:\n\n1. John "Hannibal" Smith - the leader and strategist.\n2. Templeton "Faceman" Peck - the con artist and supply specialist.\n3. B.A. Baracus - the mechanic and strongman.\n4. H.M. Murdock - the helicopter pilot and comic relief.\n\nThese four characters are the core members of The A-Team.'

### Example 2: Agent Using a Tool (Calculator)

Now let's ask a math question. Watch how the agent:
1. **Calls the model**, which returns a `tool_calls` request for `Calculator`
2. **Executes** the tool in the `tools` node
3. **Appends** the result as a `ToolMessage`
4. **Calls the model again** with that result in context
5. **Answers** without requesting another tool, ending the loop


In [11]:
# =============================================================================
# Running the Agent - Math Question (Requires Tool)
# =============================================================================
# The streamed output shows the complete message sequence:
#
# ================================ Human Message =================================
# What is 100 devided by 25?
# ================================== Ai Message ==================================
# Tool Calls:
#   Calculator (call_...)
#   Args:
#     __arg1: 100 / 25
# ================================= Tool Message =================================
# Answer: 4.0
# ================================== Ai Message ==================================
# 100 divided by 25 is 4.0.

run_agent(agent, "What is 100 devided by 25?")

# Note: Even with the typo "devided", the agent understands the intent!

================================ Human Message =================================

What is 100 devided by 25?
================================== Ai Message ==================================
Tool Calls:
  Calculator (call_meGewmePaXLQv3Es0Y0RTHO2)
 Call ID: call_meGewmePaXLQv3Es0Y0RTHO2
  Args:
    __arg1: 100 divided by 25
================================== Ai Message ==================================
Tool Calls:
  Calculator (call_meGewmePaXLQv3Es0Y0RTHO2)
 Call ID: call_meGewmePaXLQv3Es0Y0RTHO2
  Args:
    __arg1: 100 divided by 25
================================= Tool Message =================================
Name: Calculator

Answer: 4.0
================================== Ai Message ==================================

100 divided by 25 is 4.0.
================================== Ai Message ==================================

100 divided by 25 is 4.0.


{'messages': [HumanMessage(content='What is 100 devided by 25?', additional_kwargs={}, response_metadata={}, id='99b44d5a-5539-4e4f-a5ee-c264f444a075'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 83, 'total_tokens': 103, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ed36603d78', 'id': 'chatcmpl-EMeGSNsjaFvIXwLkiqjWsGxKSNcES', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a08caf-63dc-7271-bfee-200605a0c9da-0', tool_calls=[{'name': 'Calculator', 'args': {'__arg1': '100 divided by 25'}, 'id': 'call_meGewmePaXLQv

---

## Section 2: Creating Custom Tools

### Why Custom Tools?

Built-in tools are great, but real-world applications often need domain-specific functionality. You can create custom tools to:
- Query your own databases
- Call internal APIs
- Perform specialized computations
- Access proprietary data sources

### Ways to Create Custom Tools

| Method | Complexity | Use Case |
|--------|------------|----------|
| `@tool` decorator | Simple | Quick functions with no state |
| `BaseTool` subclass | Medium | When you need more control |
| `StructuredTool` | Advanced | Complex inputs with validation |

### Example: Restaurant Search Tool

Let's create a custom tool that searches a FAISS vector store containing restaurant information:

In [13]:
# =============================================================================
# STEP 1: Load the Vector Store (Knowledge Base)
# =============================================================================
# First, we need a data source for our custom tool
# Here we're loading a pre-built FAISS vector store containing restaurant FAQ data

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize the same embeddings model used to create the index
embeddings = OpenAIEmbeddings()

# Load the pre-built FAISS index from disk
# The 'index' folder contains:
#   - index.faiss: The actual vector index
#   - index.pkl: Metadata and document mappings
# 
# ⚠️ allow_dangerous_deserialization=True is required because loading
# pickled data can be a security risk if from untrusted sources
vectorstore = FAISS.load_local("vectorstore", embeddings, allow_dangerous_deserialization=True)

RuntimeError: Error in __cdecl faiss::FileIOReader::FileIOReader(const char *) at D:\a\faiss\faiss\faiss\impl\io.cpp:70: Error: 'f' failed: could not open vectorstore\index.faiss for reading: No such file or directory

In [ ]:
# =============================================================================
# STEP 2: Define a Custom Tool Using BaseTool
# =============================================================================
# Creating a custom tool by subclassing BaseTool gives you full control
# over the tool's behavior and allows for more complex logic
#
# NOTE (LangChain 1.x): the callback managers moved to langchain_core.
# `from langchain.callbacks.manager import ...` raises ModuleNotFoundError on 1.x.

from typing import Optional
from langchain_core.tools import BaseTool
from langchain_core.callbacks.manager import (
    AsyncCallbackManagerForToolRun,
    CallbackManagerForToolRun,
)


class CustomSearchTool(BaseTool):
    """
    A custom tool that searches a restaurant FAQ vector store.

    When the agent needs information about the restaurant, it will use this tool
    to retrieve relevant information from the vector store.
    """

    # REQUIRED: Tool name - used by the agent to reference this tool
    # Keep it identifier-friendly: some providers reject spaces in tool names.
    name: str = "restaurant_search"

    # REQUIRED: Description - CRITICAL! The agent uses this to decide when to use the tool
    # Make it clear and specific about what the tool does
    description: str = "useful for when you need to answer questions about our restaurant"

    def _run(self, query: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        """
        The main method that gets called when the agent uses this tool.

        Args:
            query: The search query from the agent
            run_manager: Optional callback manager for tracking

        Returns:
            String containing relevant information from the vector store
        """
        # Convert vectorstore to a retriever
        store = vectorstore.as_retriever()

        # Perform semantic search
        docs = store.invoke(query)

        # Extract text content from retrieved documents
        text_list = [doc.page_content for doc in docs]

        # Return combined results
        return "\n".join(text_list)

    async def _arun(self, query: str, run_manager: Optional[AsyncCallbackManagerForToolRun] = None) -> str:
        """
        Async version of the tool (required by BaseTool but not implemented here).
        Raise NotImplementedError if async is not supported.
        """
        raise NotImplementedError("custom_search does not support async")

In [ ]:
# =============================================================================
# STEP 3: Create an Agent with Our Custom Tool
# =============================================================================
# Now we instantiate our custom tool and create a new agent
# This agent will ONLY have access to the restaurant search tool

# Create an instance of our custom tool
# We can create multiple tools and pass them as a list
tools = [CustomSearchTool()]

# Same create_agent call as before - only the tool list changes.
# There is no agent "type" to pick: tool calling is native either way.
agent = create_agent(
    llm,
    tools,
    system_prompt=(
        "You are a helpful assistant for the Bella Vista restaurant. "
        "Use the restaurant_search tool for anything about the restaurant. "
        "For unrelated questions, answer from your own knowledge."
    ),
    middleware=[ModelCallLimitMiddleware(thread_limit=3, exit_behavior="end")],
)

### Testing Our Custom Tool Agent

#### Test 1: Question Outside the Tool's Scope

Let's first ask a question that the restaurant tool can't answer. The agent should recognize this and use its own knowledge:


In [ ]:
# =============================================================================
# Test 1: Out-of-Scope Question
# =============================================================================
# The agent only has the restaurant search tool, but this question is about
# a TV show. The agent should:
# 1. Recognize the tool won't help
# 2. Use its own knowledge to answer
#
# Notice how the agent doesn't waste time searching for irrelevant info!

run_agent(agent, "Who are the members of the A-Team?")

#### Test 2: Question Within the Tool's Scope

Now let's ask about the restaurant. The agent should use our custom tool.

---

### 🔧 Common Issue: the agent keeps searching and never answers

On 0.x this surfaced as `"Agent stopped due to iteration limit"`. On 1.x, `ModelCallLimitMiddleware` ends the run at the cap instead, so you get a truncated answer rather than an error string - but the underlying cause is the same.

**Root Cause**: the vector store returns fragmented results (questions without answers), so the model keeps calling the tool hoping for something better.

**Fixes (in order of effectiveness):**

| Fix | Where | Code Change |
|-----|-------|-------------|
| 1. Retrieve more docs | `as_retriever()` | `search_kwargs={"k": 6}` |
| 2. Better formatting | `_run()` return | Add clear headers like "Information Found:" |
| 3. Raise the call cap | `create_agent()` | `ModelCallLimitMiddleware(thread_limit=5)` |
| 4. Improve description | `description` field | Be specific about what the tool returns |

Note that fix 3 is the *least* effective of the four - a higher cap buys more attempts at a retrieval problem rather than solving it. Fixes 1, 2 and 4 change what the model actually sees.

**Below**: an improved tool that applies all four.

In [ ]:
# =============================================================================
# 🔧 FIX: Improved Custom Tool with Better Retrieval
# =============================================================================
# This improved version addresses the runaway-search issue

class ImprovedSearchTool(BaseTool):
    """Improved restaurant search tool with better retrieval and formatting."""

    name: str = "restaurant_search"

    # FIX 1: More descriptive - tells agent what to expect
    description: str = """Search Bella Vista restaurant knowledge base.
    Use for: hours, menu, reservations, ambiance, policies.
    Returns FAQ answers - extract the information needed to answer the user."""

    def _run(self, query: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        # FIX 2: Retrieve more documents (6 instead of default 4)
        store = vectorstore.as_retriever(search_kwargs={"k": 6})
        docs = store.invoke(query)

        if not docs:
            return "No information found."

        # FIX 3: Format results clearly so the model can use them directly
        results = [f"\u2022 {doc.page_content}" for doc in docs]
        return "Restaurant Information:\n" + "\n".join(results)

    async def _arun(self, query: str, run_manager=None) -> str:
        raise NotImplementedError("Async not supported")


# FIX 4: Create the agent with a higher model-call cap
improved_tools = [ImprovedSearchTool()]
improved_agent = create_agent(
    llm,
    improved_tools,
    system_prompt=(
        "You are a helpful assistant for the Bella Vista restaurant. "
        "Search the knowledge base once, then answer from what you find. "
        "Do not repeat the same search."
    ),
    middleware=[ModelCallLimitMiddleware(thread_limit=5, exit_behavior="end")],
)

In [ ]:
# =============================================================================
# Test the Improved Agent
# =============================================================================
# Run the same query with the improved agent - should get better results!

run_agent(improved_agent, "When does the restaurant open?")

### Original Agent (for comparison - may hit the model-call limit)


In [ ]:
# =============================================================================
# Test 2: In-Scope Question (Uses Restaurant Tool)
# =============================================================================
# This question should trigger the restaurant search tool
# Watch the model -> tool -> model cycle in the streamed output
#
# The agent will:
# 1. Recognize this is about the restaurant
# 2. Request the "restaurant_search" tool
# 3. Receive the results as a ToolMessage
# 4. Formulate an answer (or keep searching, hitting the cap)

run_agent(agent, "When does the restaurant open?")

---

## Section 3: Conversational Agents with Memory

### Why Add Memory?

The agents we've built so far are **stateless** - each `invoke()` starts from an empty message list. For chatbot-like applications, we need agents that:
- Remember the conversation history
- Can refer back to previous topics
- Maintain context across multiple turns

### How this works on LangChain 1.x

The 0.x `ConversationBufferMemory` class is gone (`langchain.memory` no longer exists). Persistence is now a graph concern: you pass a **checkpointer**, and every run tagged with the same `thread_id` resumes the same conversation.

| Feature | Stateless Agent | Conversational Agent |
|---------|-----------------|---------------------|
| Memory | none | `checkpointer=InMemorySaver()` |
| Context | single query | full thread, restored automatically |
| Selector | n/a | `config={"configurable": {"thread_id": "..."}}` |
| Use Case | one-off questions | multi-turn chat |

This is strictly more capable than the old `Memory` objects: swap `InMemorySaver` for a Postgres or SQLite checkpointer and the same code persists across process restarts, with one independent conversation per `thread_id`.

In [ ]:
# =============================================================================
# Creating a Conversational Agent with Memory
# =============================================================================
# This agent maintains conversation history and can reference previous messages
#
# NOTE (LangChain 1.x): langchain.memory / ConversationBufferMemory are gone.
# Conversation state lives in a checkpointer, keyed by thread_id.

from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI

# The checkpointer stores the message list after every step.
# InMemorySaver is process-local; swap in a SQLite/Postgres saver to persist.
checkpointer = InMemorySaver()

# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Create the conversational agent
# The only difference from the stateless agents above is checkpointer=
agent_chain = create_agent(
    llm,
    tools,
    system_prompt="You are a helpful conversational assistant.",
    checkpointer=checkpointer,
)

# Every call that passes this config continues the SAME conversation.
# A different thread_id starts a fresh, independent one.
config = {"configurable": {"thread_id": "demo-conversation-1"}}

In [ ]:
# =============================================================================
# Inspecting the Conversational Agent
# =============================================================================
# On 0.x you would print agent_chain.agent.llm_chain.prompt.messages to find
# the MessagesPlaceholder("chat_history") slot. There is no placeholder now -
# history isn't templated into a prompt, it IS the graph state.

print("Graph nodes:", list(agent_chain.get_graph().nodes))

# Ask the checkpointer what it currently holds for this thread.
# Before the first run this is empty - run the next cells, then re-run this one.
state = agent_chain.get_state(config)
messages = state.values.get("messages", [])
print(f"Messages stored on thread {config['configurable']['thread_id']!r}: {len(messages)}")
for m in messages:
    print(f"  {m.__class__.__name__}: {str(m.content)[:80]}")

### Demonstrating Conversational Memory

Let's have a multi-turn conversation to see how the agent maintains context. Every call below passes the same `config`, so all three turns land on the same `thread_id`. Pay attention to:
1. How the agent remembers the previous topic
2. How "there" refers back to "Paris" from the previous question - even though we never resend it
3. How the stored message list grows with each interaction


In [ ]:
# =============================================================================
# Turn 1: Initial Question
# =============================================================================
# First message in the conversation - establishes context about France/Paris
# The agent answers directly without needing to use any tools
#
# Passing `config` is what makes this a *conversation* rather than a one-off:
# the checkpointer files these messages under thread_id "demo-conversation-1".

result = run_agent(agent_chain, "What is the capital of France?", config=config)

# The final state's "messages" key holds the whole thread so far
print(f"\nConversation now has {len(result['messages'])} messages")

In [ ]:
# =============================================================================
# Turn 2: Follow-up Question (Tests Memory!)
# =============================================================================
# Here's where memory matters: "there" refers to Paris!
# We only send the new question - the checkpointer replays the earlier turns
# into the model automatically, because the thread_id matches.

result = run_agent(agent_chain, "Any suggestions what to visit there?", config=config)

# The message list keeps growing across turns
print(f"\nConversation now has {len(result['messages'])} messages")

In [ ]:
# =============================================================================
# Turn 3: Topic Switch
# =============================================================================
# The agent can handle topic changes while maintaining full conversation history
# It understands we're now asking about places to visit in China
# (based on the context of previous travel-related questions)

result = run_agent(agent_chain, "What about china", config=config)

# The full conversation history is preserved in the checkpointer.
# This is useful for:
# - Understanding user intent from context
# - Providing personalized responses
# - Maintaining coherent multi-turn dialogues
#
# Try it: re-run the inspection cell above to see all of these messages,
# or start a fresh thread_id and watch the agent lose the context.
print(f"\nFinal conversation has {len(result['messages'])} messages")

In [ ]:
# =============================================================================
# 🎯 EXERCISE: Try It Yourself!
# =============================================================================
#
# Exercise 1: Create a simple tool using the @tool decorator
#
# from langchain_core.tools import tool
#
# @tool
# def get_weather(city: str) -> str:
#     """Get the current weather for a city."""
#     # In a real app, you'd call a weather API
#     return f"The weather in {city} is sunny, 72 degrees F"
#
# Exercise 2: Add the weather tool to an agent and test it
#
#     agent = create_agent(llm, [get_weather], system_prompt="...")
#
# Exercise 3: Create an agent with multiple tools (calculator + restaurant + weather)
#
# Exercise 4: Give it a checkpointer and hold a multi-turn conversation,
#             then start a second thread_id and confirm the two stay separate.
#
# Exercise 5: Add SummarizationMiddleware alongside ModelCallLimitMiddleware
#             and watch it compress a long conversation automatically.
#
# =============================================================================


In [ ]:
# Your code here - try creating your own tools and agents!


## Summary & Key Takeaways

### What We Learned

1. **Agents** use LLMs as reasoning engines to decide which tools to use
2. **Tools** extend agent capabilities with domain-specific functions
3. **`create_agent()`** returns a compiled LangGraph graph that runs the model/tool loop
4. **Custom Tools** can be created by subclassing `BaseTool`
5. **Checkpointers** give agents memory, one conversation per `thread_id`

### Migrating from `initialize_agent`

| LangChain 0.x | LangChain 1.x |
|---------------|---------------|
| `initialize_agent(tools, llm, agent=AgentType...)` | `create_agent(llm, tools, system_prompt=...)` |
| returns `AgentExecutor` (a `Chain`) | returns `CompiledStateGraph` |
| ReAct text parsing, `handle_parsing_errors=True` | native tool calling, nothing to parse |
| `max_iterations=3` | `ModelCallLimitMiddleware(thread_limit=3)` |
| `verbose=True` | `.stream(..., stream_mode="values")` |
| `memory=ConversationBufferMemory(...)` | `checkpointer=` + `thread_id` in config |
| `invoke({"input": ...})` -> `{"output": ...}` | `invoke({"messages": [...]})` -> `{"messages": [...]}` |
| `from langchain.agents import load_tools` | `from langchain_community.agent_toolkits.load_tools import load_tools` |
| `from langchain.callbacks.manager import ...` | `from langchain_core.callbacks.manager import ...` |

### Middleware: the capability with no 0.x equivalent

`create_agent(middleware=[...])` accepts composable hooks around the model and tool calls. Shipped with LangChain 1.x:

`SummarizationMiddleware` · `HumanInTheLoopMiddleware` · `PIIMiddleware` · `ModelFallbackMiddleware` · `ModelRetryMiddleware` · `ToolRetryMiddleware` · `ToolErrorMiddleware` · `ModelCallLimitMiddleware` · `ToolCallLimitMiddleware` · `LLMToolSelectorMiddleware` · `ContextEditingMiddleware` · `TodoListMiddleware`

Write your own with `@before_model`, `@after_model`, `@wrap_model_call`, `@wrap_tool_call`, or `@dynamic_prompt`.

### Best Practices

1. **Tool Descriptions Matter**: write clear, specific descriptions - this is still the single biggest lever on agent behaviour
2. **Keep tool names identifier-safe**: some providers reject spaces in tool names
3. **Set Limits**: `ModelCallLimitMiddleware` prevents runaway loops
4. **Prefer fixing retrieval over raising limits**: a higher cap hides a retrieval problem, it doesn't solve it

### What's Next?

- Explore **LangGraph** directly for agent workflows that aren't a simple tool loop
- Learn about **structured output** with `create_agent(response_format=...)`
- Build **Multi-Agent Systems** by composing agent graphs as nodes

---

*For the latest best practices, check the [LangChain Agents documentation](https://docs.langchain.com/oss/python/langchain/agents)*